# 📊 CAT Data Lake — Row Count & Raw String Validator

**Purpose:** Source file (local) aur Trino (Silver) ka row count compare karna — total + event-wise. Raw string validation bhi.

| Source | Detail |
|--------|--------|
| **Source File** | Local `.bz2` file (CSV ya JSON) |
| **Destination** | Silver layer → Trino query |

### Checks Performed
| # | Check | Description |
|---|-------|-------------|
| 1 | **Total Row Count** | Source total rows vs Trino total rows |
| 2 | **Event-wise Row Count** | Per event type count match (MENO, MEOR, MEOC...) |
| 3 | **Missing / Extra Events** | Koi event source mein hai Trino mein nahi ya vice versa |
| 4 | **Raw String Validation** | Trino ki `raw_str` column = source row ka comma-joined string? |

### File Type Handling
| File Type | Event Type | Raw String |
|-----------|-----------|------------|
| CSV `.bz2` | Column index 3 (0-based) | Comma-joined row values |
| JSON `.bz2` | `type` key | JSON string of object |

---
## 📦 Cell 1 — Libraries

In [24]:
import bz2
import csv
import json
import urllib3
import pandas as pd
from collections import Counter
from IPython.display import display
from trino.dbapi import connect
from trino.auth import BasicAuthentication

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 200)

print('✅ Libraries loaded!')

✅ Libraries loaded!


---
## ⚙️ Cell 2 — Configuration (Sirf yahan change karo)

In [25]:
# ─────────────────────────────────────────────────────────────────
#  SOURCE FILE (local path)
# ─────────────────────────────────────────────────────────────────
SOURCE_FILE = r"/home/shariq/Downloads/CATFILE-VLVT/96547_VCLT_20260427_OrderEvents_000001.csv.bz2"
# FILE_TYPE: 'csv' ya 'json' — filename se auto detect hoga, manually override bhi kar sakte ho
FILE_TYPE = None   # None = auto detect

# CSV specific
CSV_EVENT_TYPE_COL = 3   # 0-based index — event type kaunse column mein hai

# JSON specific
JSON_EVENT_TYPE_KEY = 'type'   # JSON mein event type ki key

# ─────────────────────────────────────────────────────────────────
#  TRINO CONNECTION
# ─────────────────────────────────────────────────────────────────
TRINO_CONFIG = {
    "host"    : "3.221.185.123",
    "port"    : 8443,
    "user"    : "shariq.mehmood",
    "password": "Prometheus-X!",
    "catalog" : "iceberg",
    "schema"  : "ignite_test"
}

TARGET_TABLE = 'iceberg.qa.silver_catevents'
WHERE_CLAUSE = "s3_path like '%s3://reportingfilesandbackup/CATManagementFiles/TRAFix/VCLT/CATReports/96547_VCLT_20260427_OrderEvents_000001.csv.bz2%'"

# ─────────────────────────────────────────────────────────────────
#  RAW STRING VALIDATION
# ─────────────────────────────────────────────────────────────────
# Kitni rows raw_str validate karni hain (sample)
# None = saari rows validate karo (slow ho sakta hai)
RAW_STR_SAMPLE_SIZE = 1000

# ── Run to check null values──
SCHEMA_FILE_PATH = r"/home/shariq/Downloads/data_configurator_public_cat_event_schema.csv"
NULL_SAMPLE_SIZE = 5000   # source aur Trino dono se same size

print(f'✅ Source File  : {SOURCE_FILE}')
print(f'✅ Target Table : {TARGET_TABLE}')
print(f'✅ Null check Sample   : {NULL_SAMPLE_SIZE} rows')
print(f'✅ Rawstring Sample   : {RAW_STR_SAMPLE_SIZE} rows')

✅ Source File  : /home/shariq/Downloads/CATFILE-VLVT/96547_VCLT_20260427_OrderEvents_000001.csv.bz2
✅ Target Table : iceberg.qa.silver_catevents
✅ Null check Sample   : 5000 rows
✅ Rawstring Sample   : 1000 rows


---
## 📂 Cell 3 — Source File Read karo

In [26]:
def detect_file_type(path):
    """Filename se file type detect karo."""
    name = path.lower()
    if '_json.bz2' in name or name.endswith('.json.bz2') or name.endswith('.json'):
        return 'json'
    return 'csv'


def read_source_file(path, file_type=None):
    """
    Source file read karo (.bz2 ya plain).
    Returns:
        file_type    : 'csv' ya 'json'
        rows         : list of rows
                       CSV  → list of lists (values)
                       JSON → list of dicts
        event_counts : Counter {event_type: count}
        raw_strings  : list of raw strings (comma-joined for CSV, json str for JSON)
    """
    ftype = file_type or detect_file_type(path)
    is_bz2 = path.endswith('.bz2')

    open_fn = bz2.open if is_bz2 else open

    rows        = []
    raw_strings = []
    event_counts = Counter()

    print(f'⏳ Reading {ftype.upper()} file ({'bz2 compressed' if is_bz2 else 'plain'})...')

    if ftype == 'csv':
        with open_fn(path, 'rt', encoding='utf-8') as f:
            reader = csv.reader(f)
            for row in reader:
                if not row:   # empty line skip
                    continue
                rows.append(row)
                raw_strings.append(','.join(row))
                if len(row) > CSV_EVENT_TYPE_COL:
                    event_counts[row[CSV_EVENT_TYPE_COL]] += 1
                else:
                    event_counts['__UNKNOWN__'] += 1

    elif ftype == 'json':
        with open_fn(path, 'rt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                rows.append(obj)
                raw_strings.append(json.dumps(obj, separators=(',', ':')))
                event_type = obj.get(JSON_EVENT_TYPE_KEY, '__UNKNOWN__')
                event_counts[event_type] += 1

    print(f'\n✅ Source file read complete!')
    print(f'   📊 Total rows        : {len(rows)}')
    print(f'   🗂️  Unique event types : {len(event_counts)}')
    print(f'\n   Event-wise breakdown (SOURCE):')
    for evt, cnt in sorted(event_counts.items()):
        print(f'      {evt:<12} : {cnt}')

    return ftype, rows, event_counts, raw_strings


file_type, source_rows, source_event_counts, source_raw_strings = read_source_file(SOURCE_FILE, FILE_TYPE)

⏳ Reading CSV file (bz2 compressed)...

✅ Source file read complete!
   📊 Total rows        : 455516
   🗂️  Unique event types : 10

   Event-wise breakdown (SOURCE):
      MECR         : 6129
      MEMR         : 203388
      MENO         : 18226
      MEOC         : 6126
      MEOM         : 203388
      MEOR         : 18255
      MOMR         : 1
      MONO         : 1
      MOOM         : 1
      MOOR         : 1


---
## 🔌 Cell 4 — Trino se Row Count Fetch karo

In [27]:
def get_trino_connection():
    auth = BasicAuthentication(TRINO_CONFIG['user'], TRINO_CONFIG['password'])
    return connect(
        host        = TRINO_CONFIG['host'],
        port        = TRINO_CONFIG['port'],
        user        = TRINO_CONFIG['user'],
        auth        = auth,
        http_scheme = 'https',
        verify      = False,
        catalog     = TRINO_CONFIG['catalog'],
        schema      = TRINO_CONFIG['schema']
    )


def fetch_trino_counts():
    """
    Trino se event-wise row count aur total count fetch karo.
    Returns: (total_count, event_counts_dict)
    """
    conn = get_trino_connection()
    cur  = conn.cursor()

    # Event-wise count
    query_event = f"""
    SELECT "type", COUNT(*) as cnt
    FROM {TARGET_TABLE}
    WHERE {WHERE_CLAUSE}
    GROUP BY "type"
    ORDER BY "type"
"""
    print('⏳ Trino se event-wise count fetch ho raha hai...')
    cur.execute(query_event)
    trino_event_counts = Counter({row[0]: row[1] for row in cur.fetchall()})

    total = sum(trino_event_counts.values())

    print(f'\n✅ Trino count fetch complete!')
    print(f'   📊 Total rows        : {total}')
    print(f'   🗂️  Unique event types : {len(trino_event_counts)}')
    print(f'\n   Event-wise breakdown (TRINO):')
    for evt, cnt in sorted(trino_event_counts.items()):
        print(f'      {evt:<12} : {cnt}')

    return total, trino_event_counts


trino_total, trino_event_counts = fetch_trino_counts()

⏳ Trino se event-wise count fetch ho raha hai...

✅ Trino count fetch complete!
   📊 Total rows        : 455516
   🗂️  Unique event types : 10

   Event-wise breakdown (TRINO):
      MECR         : 6129
      MEMR         : 203388
      MENO         : 18226
      MEOC         : 6126
      MEOM         : 203388
      MEOR         : 18255
      MOMR         : 1
      MONO         : 1
      MOOM         : 1
      MOOR         : 1


---
## 🔢 Cell 5 — Row Count Comparison

In [28]:
source_total = len(source_rows)

print('=' * 70)
print('📊 ROW COUNT COMPARISON')
print('=' * 70)

# ── Total count ──
total_match = source_total == trino_total
total_icon  = '✅' if total_match else '❌'
print(f'\n{total_icon} TOTAL ROWS')
print(f'   Source : {source_total}')
print(f'   Trino  : {trino_total}')
if not total_match:
    diff = trino_total - source_total
    print(f'   ❌ Difference : {diff:+d} ({"extra in Trino" if diff > 0 else "missing in Trino"})')

# ── Event-wise count ──
print(f'\n{"-" * 70}')
print(f'  {"EVENT TYPE":<14} | {"SOURCE":>8} | {"TRINO":>8} | {"DIFF":>8} | STATUS')
print(f'{"-" * 70}')

all_events    = sorted(set(source_event_counts.keys()) | set(trino_event_counts.keys()))
event_results = []
all_match     = True

for evt in all_events:
    src_cnt   = source_event_counts.get(evt, 0)
    trn_cnt   = trino_event_counts.get(evt, 0)
    diff      = trn_cnt - src_cnt
    match     = src_cnt == trn_cnt
    if not match:
        all_match = False

    # Status label
    if match:
        status = '✅ MATCH'
    elif src_cnt == 0:
        status = '🔵 EXTRA IN TRINO'
    elif trn_cnt == 0:
        status = '🔴 MISSING IN TRINO'
    else:
        status = f'❌ MISMATCH ({diff:+d})'

    print(f'  {evt:<14} | {src_cnt:>8} | {trn_cnt:>8} | {diff:>+8} | {status}')
    event_results.append({'event_type': evt, 'source_count': src_cnt,
                          'trino_count': trn_cnt, 'diff': diff, 'status': status})

print(f'{"-" * 70}')
print(f'  {"TOTAL":<14} | {source_total:>8} | {trino_total:>8} | {trino_total - source_total:>+8} | {"✅ MATCH" if total_match else "❌ MISMATCH"}')
print(f'=' * 70)

if all_match and total_match:
    print('\n✅ ALL COUNTS MATCH — Source aur Trino mein row count bilkul same hai!')
else:
    print('\n❌ COUNT MISMATCH FOUND — Details upar table mein dekho!')

row_count_df = pd.DataFrame(event_results)

📊 ROW COUNT COMPARISON

✅ TOTAL ROWS
   Source : 455516
   Trino  : 455516

----------------------------------------------------------------------
  EVENT TYPE     |   SOURCE |    TRINO |     DIFF | STATUS
----------------------------------------------------------------------
  MECR           |     6129 |     6129 |       +0 | ✅ MATCH
  MEMR           |   203388 |   203388 |       +0 | ✅ MATCH
  MENO           |    18226 |    18226 |       +0 | ✅ MATCH
  MEOC           |     6126 |     6126 |       +0 | ✅ MATCH
  MEOM           |   203388 |   203388 |       +0 | ✅ MATCH
  MEOR           |    18255 |    18255 |       +0 | ✅ MATCH
  MOMR           |        1 |        1 |       +0 | ✅ MATCH
  MONO           |        1 |        1 |       +0 | ✅ MATCH
  MOOM           |        1 |        1 |       +0 | ✅ MATCH
  MOOR           |        1 |        1 |       +0 | ✅ MATCH
----------------------------------------------------------------------
  TOTAL          |   455516 |   455516 |       +0 | 

---
## 🧵 Cell 6 — Raw String Validation

In [29]:
def fetch_trino_rawstrings(sample_size=None):
    """
    Trino se raw_str column fetch karo.
    sample_size = None means fetch all.
    Returns list of raw_str values from Trino.
    """
    conn = get_trino_connection()
    cur  = conn.cursor()

    limit_clause = f'LIMIT {sample_size}' if sample_size else ''
    query = f"""
        SELECT raw_str
        FROM {TARGET_TABLE}
        WHERE {WHERE_CLAUSE}
        {limit_clause}
    """
    print(f'⏳ Trino se raw_str fetch ho raha hai ({sample_size if sample_size else "all"} rows)...')
    cur.execute(query)
    results = [row[0] for row in cur.fetchall()]
    print(f'✅ Fetched {len(results)} raw_str values from Trino')
    return results


def validate_raw_strings(source_raws, trino_raws, file_type):
    """
    Source raw strings vs Trino raw_str compare karo.
    Note: Order match guarantee nahi hoti (Trino unordered) —
    isliye set-based comparison + sample spot check dono karenge.
    """
    print('\n' + '=' * 70)
    print('🧵 RAW STRING VALIDATION')
    print('=' * 70)
    print(f'   File type     : {file_type.upper()}')
    print(f'   Source rows   : {len(source_raws)}')
    print(f'   Trino rows    : {len(trino_raws)}')

    # ── Null check in Trino raw_str ──
    trino_nulls = sum(1 for r in trino_raws if not r or str(r).strip() in {'', 'None', 'null', 'nan'})
    print(f'\n   🔍 Null raw_str in Trino : {trino_nulls}')
    if trino_nulls > 0:
        print(f'   ❌ {trino_nulls} rows mein raw_str NULL hai — ingestion issue!')
    else:
        print(f'   ✅ Koi null raw_str nahi')

    # ── Set-based match ──
    source_set = set(source_raws)
    trino_set  = set(str(r) for r in trino_raws if r)

    in_source_not_trino = source_set - trino_set
    in_trino_not_source = trino_set - source_set

    print(f'\n   📊 Set-based comparison (unique raw strings):')
    print(f'      Source unique  : {len(source_set)}')
    print(f'      Trino unique   : {len(trino_set)}')

    if not in_source_not_trino and not in_trino_not_source:
        print(f'\n   ✅ PASS — Sab raw strings match karte hain!')
    else:
        if in_source_not_trino:
            print(f'\n   🔴 Source mein hai, Trino mein NAHI ({len(in_source_not_trino)}):')  
            for s in list(in_source_not_trino)[:5]:
                print(f'      → {s[:120]}')
            if len(in_source_not_trino) > 5:
                print(f'      ... aur {len(in_source_not_trino) - 5} aur')

        if in_trino_not_source:
            print(f'\n   🔵 Trino mein hai, Source mein NAHI ({len(in_trino_not_source)}):')
            for s in list(in_trino_not_source)[:5]:
                print(f'      → {str(s)[:120]}')
            if len(in_trino_not_source) > 5:
                print(f'      ... aur {len(in_trino_not_source) - 5} aur')

    # ── Sample spot check ──
    print(f'\n   🔎 Sample Spot Check (first 5 source rows vs Trino):')
    print(f'   {"─" * 60}')
    for i, src_raw in enumerate(source_raws[:5]):
        found = str(src_raw) in trino_set
        icon  = '✅' if found else '❌'
        print(f'   {icon} Row {i+1}: {str(src_raw)[:100]}...')
        if not found:
            print(f'      ↳ NOT FOUND in Trino raw_str')

    return {
        'trino_nulls'         : trino_nulls,
        'in_source_not_trino' : len(in_source_not_trino),
        'in_trino_not_source' : len(in_trino_not_source),
        'raw_str_pass'        : trino_nulls == 0 and not in_source_not_trino and not in_trino_not_source
    }


trino_raw_strings = fetch_trino_rawstrings(RAW_STR_SAMPLE_SIZE)
raw_str_results   = validate_raw_strings(
    source_raws = source_raw_strings[:RAW_STR_SAMPLE_SIZE] if RAW_STR_SAMPLE_SIZE else source_raw_strings,
    trino_raws  = trino_raw_strings,
    file_type   = file_type
)

⏳ Trino se raw_str fetch ho raha hai (1000 rows)...
✅ Fetched 1000 raw_str values from Trino

🧵 RAW STRING VALIDATION
   File type     : CSV
   Source rows   : 1000
   Trino rows    : 1000

   🔍 Null raw_str in Trino : 0
   ✅ Koi null raw_str nahi

   📊 Set-based comparison (unique raw strings):
      Source unique  : 1000
      Trino unique   : 1000

   ✅ PASS — Sab raw strings match karte hain!

   🔎 Sample Spot Check (first 5 source rows vs Trino):
   ────────────────────────────────────────────────────────────
   ✅ Row 1: NEW,,20260427_20260427073430_3338437_1,MEOR,VCLT,20260225T000000,20260225VCLT15918,COPX,,20260427 07...
   ✅ Row 2: NEW,,20260427_20260427073430_3400214_1,MEOR,VCLT,20260224T000000,20260224VCLT216847,IBIT,,20260427 0...
   ✅ Row 3: NEW,,20260427_20260427073430_3529242_1,MEOR,VCLT,20260218T000000,20260218VCLT160542,AAL,,20260427 07...
   ✅ Row 4: NEW,,20260427_20260427073430_3529256_1,MEOR,VCLT,20260218T000000,20260218VCLT160798,AAL,,20260427 07...
   ✅ Row 5: NEW,

---
## 🧵 Cell 9 — Raw String: Trino Fetch + Compare
Trino se actual `raw_str` values fetch karo aur source se compare karo.
Agar trailing comma mismatch ho toh `STRIP_TRAILING` mode auto-detect karta hai.

In [30]:
def fetch_trino_rawstr_sample(sample_size=50):
    """
    Trino se raw_str fetch karo.
    Returns: list of (type, raw_str) tuples
    """
    conn = get_trino_connection()
    cur  = conn.cursor()
    query = f"""
        SELECT type, raw_str
        FROM {TARGET_TABLE}
        WHERE {WHERE_CLAUSE}
        LIMIT {sample_size}
    """
    print(f'⏳ Trino se raw_str fetch ho raha hai ({sample_size} rows)...')
    cur.execute(query)
    results = cur.fetchall()
    print(f'✅ {len(results)} rows fetched from Trino')
    return results


def build_source_rawstr_map(source_rows, file_type):
    """
    Source rows se raw_str map banao — do versions:
    1. As-is  (trailing empty fields samet)
    2. Stripped (trailing empty fields hata k)
    Returns: {as_is_str: stripped_str}
    """
    result = {}
    if file_type == 'csv':
        for row in source_rows:
            as_is   = ','.join(row)
            # Trailing empty fields strip karo
            stripped_row = row.copy()
            while stripped_row and stripped_row[-1].strip() == '':
                stripped_row.pop()
            stripped = ','.join(stripped_row)
            result[as_is] = stripped
    elif file_type == 'json':
        for row in source_rows:
            as_is   = json.dumps(row, separators=(',', ':'))
            stripped = as_is   # JSON mein trailing strip concept nahi
            result[as_is] = stripped
    return result


def compare_rawstrings(trino_rows, source_rawstr_map):
    """
    Trino raw_str vs source compare karo.
    Auto-detect karta hai k Trino trailing commas rakhta hai ya strip karta hai.
    """
    print('\n' + '=' * 80)
    print('🧵 RAW STRING COMPARISON — TRINO vs SOURCE')
    print('=' * 80)

    as_is_set    = set(source_rawstr_map.keys())
    stripped_set = set(source_rawstr_map.values())

    trino_rawstrs = [str(r[1]) if r[1] is not None else '' for r in trino_rows]
    trino_types   = [str(r[0]) for r in trino_rows]

    # ── Null check ──
    null_count = sum(1 for r in trino_rawstrs if not r or r.strip() in {'', 'None', 'null', 'nan'})
    print(f'\n🔍 NULL raw_str in Trino : {null_count}', end=' ')
    print('✅' if null_count == 0 else '❌ — ingestion issue!')

    # ── Auto detect: trailing comma mode ──
    as_is_matches    = sum(1 for r in trino_rawstrs if r in as_is_set)
    stripped_matches = sum(1 for r in trino_rawstrs if r in stripped_set)
    total            = len(trino_rawstrs)

    print(f'\n🔎 Auto-detect trailing comma mode:')
    print(f'   As-is match    (with trailing commas) : {as_is_matches}/{total}')
    print(f'   Stripped match (without trailing)     : {stripped_matches}/{total}')

    if as_is_matches >= stripped_matches:
        mode        = 'AS-IS'
        compare_set = as_is_set
    else:
        mode        = 'STRIPPED'
        compare_set = stripped_set

    print(f'\n   ✅ Mode selected: {mode} — Trino raw_str {"trailing commas ke saath store hai" if mode == "AS-IS" else "trailing commas strip karke store hai"}')

    # ── Per row comparison ──
    print(f'\n{"-" * 80}')
    print(f'  {"ROW":>4} | {"EVENT":>6} | {"STATUS":<10} | RAW_STR (first 80 chars)')
    print(f'{"-" * 80}')

    match_count    = 0
    mismatch_count = 0
    mismatch_rows  = []

    for i, (evt, raw) in enumerate(zip(trino_types, trino_rawstrs), 1):
        if not raw or raw.strip() in {'', 'None', 'null'}:
            status = '❌ NULL'
            mismatch_count += 1
        elif raw in compare_set:
            status = '✅ MATCH'
            match_count += 1
        else:
            status = '❌ MISMATCH'
            mismatch_count += 1
            mismatch_rows.append({'row': i, 'event': evt, 'trino_raw': raw[:200]})

        print(f'  {i:>4} | {evt:>6} | {status:<10} | {str(raw)[:80]}')

    print(f'{"-" * 80}')
    print(f'  ✅ MATCH     : {match_count}')
    print(f'  ❌ MISMATCH  : {mismatch_count}')
    print(f'  Mode used    : {mode}')
    print('=' * 80)

    if mismatch_rows:
        print(f'\n❌ Mismatch detail:')
        display(pd.DataFrame(mismatch_rows))

    return mode, match_count, mismatch_count


# ── Run ──
source_rawstr_map = build_source_rawstr_map(source_rows, file_type)
trino_sample_rows = fetch_trino_rawstr_sample(sample_size=RAW_STR_SAMPLE_SIZE)
raw_mode, raw_match, raw_mismatch = compare_rawstrings(trino_sample_rows, source_rawstr_map)


⏳ Trino se raw_str fetch ho raha hai (1000 rows)...
✅ 1000 rows fetched from Trino

🧵 RAW STRING COMPARISON — TRINO vs SOURCE

🔍 NULL raw_str in Trino : 0 ✅

🔎 Auto-detect trailing comma mode:
   As-is match    (with trailing commas) : 1000/1000
   Stripped match (without trailing)     : 920/1000

   ✅ Mode selected: AS-IS — Trino raw_str trailing commas ke saath store hai

--------------------------------------------------------------------------------
   ROW |  EVENT | STATUS     | RAW_STR (first 80 chars)
--------------------------------------------------------------------------------
     1 |   MEOR | ✅ MATCH    | NEW,,20260427_20260427073437_4350251_1,MEOR,VCLT,20260409T000000,20260409VCLT221
     2 |   MEOR | ✅ MATCH    | NEW,,20260427_20260427073430_3400214_1,MEOR,VCLT,20260224T000000,20260224VCLT216
     3 |   MEOR | ✅ MATCH    | NEW,,20260427_20260427073430_4217782_1,MEOR,VCLT,20260406T000000,20260406VCLT33,
     4 |   MEOR | ✅ MATCH    | NEW,,20260427_20260427073430_4217783_1

---
## 📤 Cell — Script Raw Strings Export (Top 50)
Script ki banai hui top 50 raw strings print karo — Excel mein 3rd column k liye.

In [13]:
# Script ki banai hui raw strings — top 50
# Ye wahi strings hain jo Cell 9 mein Trino se compare ki gayi hain

print('📤 SCRIPT RAW STRINGS — TOP 50')
print('(Excel mein 3rd column mein paste karo)')
print('=' * 80)

for i, raw in enumerate(source_raw_strings[:50], 1):
    print(f'{raw}')


📤 SCRIPT RAW STRINGS — TOP 50
(Excel mein 3rd column mein paste karo)
{"actionType":"NEW","firmROEID":"20260422_2d3620c7-118-004i_MENO","type":"MENO","CATReporterIMID":"VCGO","orderKeyDate":"20260422T161506.222000","orderID":"2d3620c7-118-004i","symbol":"NOWL","eventTimestamp":"20260422T161506.222000","manualFlag":false,"electronicDupFlag":false,"deptType":"A","solicitationFlag":false,"side":"B","price":4.19,"quantity":25000,"orderType":"LMT","timeInForce":{"DAY":20260423},"tradingSession":"ALL","handlingInstructions":{"NH":true},"custDspIntrFlag":false,"firmDesignatedID":"P006900000000000000","accountHolderType":"A","affiliateFlag":false,"negotiatedTradeFlag":false,"representativeInd":"N"}
{"actionType":"NEW","firmROEID":"20260422_2d3620c7-118-004i_MEOR","type":"MEOR","CATReporterIMID":"VCGO","orderKeyDate":"20260422T161506.222000","orderID":"2d3620c7-118-004i","symbol":"NOWL","eventTimestamp":"20260422T161506.222000","manualFlag":false,"electronicDupFlag":false,"senderIMID":"126588:V

---
## 🔬 Cell 10 — Null Validation: True NULL ya Source Empty?
Trino mein jo columns NULL dikh rahe hain — kya woh source mein bhi empty the?
Agar source mein value thi aur Trino mein NULL hai → **ingestion bug**.
Agar source mein bhi empty tha → **expected NULL**.

In [31]:
import pandas as pd

def fetch_trino_full_sample(sample_size=50):
    """
    Trino se saari columns fetch karo (NULL check k liye).
    """
    conn = get_trino_connection()
    cur  = conn.cursor()
    query = f"""
        SELECT *
        FROM {TARGET_TABLE}
        WHERE {WHERE_CLAUSE}
        LIMIT {sample_size}
    """
    print(f'⏳ Trino se full data fetch ho raha hai ({sample_size} rows)...')
    cur.execute(query)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    print(f'✅ {len(df)} rows, {len(df.columns)} columns fetched')
    return df


def build_source_df(source_rows, file_type, schema_file):
    """
    Source rows ko schema k positionkey se column names map karke DataFrame banao.
    Multi-event file hai — har row ka event type dekh k uski schema use karo.
    """
    schema = pd.read_csv(schema_file)
    schema.columns = schema.columns.str.strip()
    schema['fieldname'] = schema['fieldname'].str.strip().str.lower()
    schema['eventtype'] = schema['eventtype'].str.strip()

    # Per event: positionkey -> fieldname map (0-based index = positionkey - 1)
    event_pos_map = {}
    for evt, grp in schema.groupby('eventtype'):
        event_pos_map[evt] = {int(r['positionkey']) - 1: r['fieldname']
                              for _, r in grp.iterrows()}

    records = []
    if file_type == 'csv':
        for row in source_rows:
            if len(row) <= CSV_EVENT_TYPE_COL:
                continue
            evt     = row[CSV_EVENT_TYPE_COL]
            pos_map = event_pos_map.get(evt, {})
            record  = {'type': evt}
            for pos_idx, fieldname in pos_map.items():
                record[fieldname] = row[pos_idx] if pos_idx < len(row) else ''
            records.append(record)

    elif file_type == 'json':
        for obj in source_rows:
            record = {k.lower(): v for k, v in obj.items()}
            records.append(record)

    return pd.DataFrame(records).fillna('')


def validate_nulls(trino_df, source_df):
    """
    Trino mein NULL values ko source se validate karo.
    Har column k liye:
      - Trino NULL count
      - Source empty count (same rows)
      - True NULL  : source bhi empty tha  → expected
      - False NULL : source mein value thi  → ingestion bug
    """
    print('\n' + '=' * 80)
    print('🔬 NULL VALIDATION — TRUE NULL vs INGESTION BUG')
    print('=' * 80)
    print('  Legend:')
    print('  ✅ TRUE NULL   — source mein bhi empty tha (expected)')
    print('  ❌ FALSE NULL  — source mein value thi, Trino mein NULL (ingestion bug)')
    print('  ⏭️  SKIP        — Trino mein koi NULL nahi')
    print()

    # Common columns only
    trino_cols  = set(trino_df.columns.str.lower())
    source_cols = set(source_df.columns.str.lower())
    common_cols = sorted(trino_cols & source_cols - {'raw_str', 's3_path',
                         'ingestedfordate', 'keydate', 'boothid', 'sourcefile',
                         'submitterid', 'vendor'})

    results = []
    false_null_cols = []

    print(f'  {"COLUMN":<35} | {"TRINO NULL":>10} | {"SRC EMPTY":>10} | {"FALSE NULL":>10} | STATUS')
    print(f'  {"-" * 80}')

    for col in common_cols:
        trino_col  = trino_df[col] if col in trino_df.columns else trino_df[trino_df.columns[trino_df.columns.str.lower() == col][0]]
        source_col = source_df[col] if col in source_df.columns else source_df[source_df.columns[source_df.columns.str.lower() == col][0]]

        # Align by index (both are sample, same order assumed)
        min_len = min(len(trino_col), len(source_col))
        t_vals  = trino_col.iloc[:min_len]
        s_vals  = source_col.iloc[:min_len]

        trino_null_mask  = t_vals.isna() | t_vals.astype(str).str.strip().isin(['', 'None', 'null', 'nan', 'NaN'])
        source_empty_mask = s_vals.astype(str).str.strip() == ''

        trino_null_count  = trino_null_mask.sum()
        source_empty_count = source_empty_mask.sum()

        # False NULL: Trino NULL hai BUT source mein value thi
        false_null_mask  = trino_null_mask & ~source_empty_mask
        false_null_count = false_null_mask.sum()

        if trino_null_count == 0:
            status = '⏭️  SKIP'
        elif false_null_count > 0:
            status = f'❌ FALSE NULL'
            false_null_cols.append(col)
        else:
            status = '✅ TRUE NULL'

        if trino_null_count > 0:   # sirf nulls wale columns print karo
            print(f'  {col:<35} | {trino_null_count:>10} | {source_empty_count:>10} | {false_null_count:>10} | {status}')

        results.append({'column': col, 'trino_null': trino_null_count,
                         'source_empty': source_empty_count,
                         'false_null': false_null_count, 'status': status})

    print(f'  {"-" * 80}')

    if false_null_cols:
        print(f'\n❌ FALSE NULL columns ({len(false_null_cols)}) — ingestion bug likely:')
        for c in false_null_cols:
            print(f'   → {c}')
    else:
        print('\n✅ Koi FALSE NULL nahi — sab nulls source mein bhi empty the!')

    return pd.DataFrame(results)



trino_full_df  = fetch_trino_full_sample(NULL_SAMPLE_SIZE)
source_df      = build_source_df(source_rows[:NULL_SAMPLE_SIZE], file_type, SCHEMA_FILE_PATH)
null_results_df = validate_nulls(trino_full_df, source_df)


⏳ Trino se full data fetch ho raha hai (5000 rows)...
✅ 5000 rows, 121 columns fetched

🔬 NULL VALIDATION — TRUE NULL vs INGESTION BUG
  Legend:
  ✅ TRUE NULL   — source mein bhi empty tha (expected)
  ❌ FALSE NULL  — source mein value thi, Trino mein NULL (ingestion bug)
  ⏭️  SKIP        — Trino mein koi NULL nahi

  COLUMN                              | TRINO NULL |  SRC EMPTY | FALSE NULL | STATUS
  --------------------------------------------------------------------------------
  accountholdertype                   |       4942 |       4942 |          0 | ✅ TRUE NULL
  affiliateflag                       |       2434 |       2434 |          1 | ❌ FALSE NULL
  aggregatedorders                    |       5000 |       5000 |          0 | ✅ TRUE NULL
  atsdisplayind                       |       5000 |       5000 |          0 | ✅ TRUE NULL
  atsordertype                        |       5000 |       5000 |          0 | ✅ TRUE NULL
  cancelqty                           |       4987 |    

---
## 📋 Cell 7 — Final Summary

In [15]:
print('\n' + '=' * 70)
print('📋 FINAL SUMMARY')
print('=' * 70)

checks = [
    ('Total Row Count',    '✅ PASS' if total_match else '❌ FAIL',
     f'Source={source_total}, Trino={trino_total}'),

    ('Event-wise Count',   '✅ PASS' if all_match else '❌ FAIL',
     f'{sum(1 for r in event_results if "MATCH" in r["status"])}/{len(event_results)} events match'),

    ('Raw String NULLs',   '✅ PASS' if raw_str_results['trino_nulls'] == 0 else '❌ FAIL',
     f"{raw_str_results['trino_nulls']} null raw_str in Trino"),

    ('Raw String Match',   '✅ PASS' if raw_str_results['raw_str_pass'] else '❌ FAIL',
     f"Source-only: {raw_str_results['in_source_not_trino']}, Trino-only: {raw_str_results['in_trino_not_source']}"),
]

print(f'\n  {"CHECK":<25} | {"STATUS":<12} | DETAIL')
print(f'  {"-" * 65}')
for check, status, detail in checks:
    print(f'  {check:<25} | {status:<12} | {detail}')

print(f'\n  Source File : {SOURCE_FILE}')
print(f'  Table       : {TARGET_TABLE}')
print('=' * 70)

overall_pass = all(status == '✅ PASS' for _, status, _ in checks)
if overall_pass:
    print('\n🎉 ALL CHECKS PASSED!')
else:
    print('\n❌ SOME CHECKS FAILED — Details upar dekho!')


📋 FINAL SUMMARY

  CHECK                     | STATUS       | DETAIL
  -----------------------------------------------------------------
  Total Row Count           | ✅ PASS       | Source=494, Trino=494
  Event-wise Count          | ✅ PASS       | 13/13 events match
  Raw String NULLs          | ✅ PASS       | 0 null raw_str in Trino
  Raw String Match          | ❌ FAIL       | Source-only: 494, Trino-only: 494

  Source File : /home/shariq/Downloads/93012_VCGO_20260423_G2_OrderEvents_000001.json.bz2
  Table       : iceberg.qa.silver_catevents

❌ SOME CHECKS FAILED — Details upar dekho!


---
## 🔎 Cell 8 — Debug: Raw String Mismatch Detail (Optional)

In [ ]:
# Agar raw string mismatch aaya ho toh yahan specific rows inspect karo
# Konsi source row Trino mein match nahi hui

trino_set_full = set(str(r) for r in trino_raw_strings if r)

mismatch_rows = []
check_raws = source_raw_strings[:RAW_STR_SAMPLE_SIZE] if RAW_STR_SAMPLE_SIZE else source_raw_strings

for i, raw in enumerate(check_raws):
    if str(raw) not in trino_set_full:
        mismatch_rows.append({'source_row_index': i + 1, 'source_raw': str(raw)[:200]})

if mismatch_rows:
    print(f'❌ Raw string mismatches found: {len(mismatch_rows)}')
    display(pd.DataFrame(mismatch_rows))
else:
    print('✅ Koi raw string mismatch nahi — sab checked rows Trino mein hain!')